In [1]:
import os

print(os.getenv("GROQ_API_KEY") is not None)

True


In [2]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

response = model.invoke("Hello, how are you?")

print(response.content)

Hello! I’m doing great—thanks for asking. How can I help you today?


In [9]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage
import operator

from langgraph.checkpoint.memory import InMemorySaver

In [10]:
load_dotenv(dotenv_path=".env")

True

In [11]:
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [12]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [13]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = model.invoke(prompt).content

    return {'joke': response}

In [14]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = model.invoke(prompt).content

    return {'explanation': response}

In [15]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [16]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to get a *crust* of a living! 🍕😄',
 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Personification** | The pizza is treated like a human who can *apply for a job*. | It’s absurd to imagine a slice of dough with cheese and toppings doing a job‑hunt. That absurdity sets the stage for the punchline. |\n| **Wordplay on “crust”** | “Crust” is the outer edge of a pizza. | The joke hinges on the double meaning of the word. |\n| **Play on “cost” / “crust”** | The phrase “a crust of a living” sounds like “a cost of a living” (i.e., the money you need to live). | The pun is that the pizza wants a *crust* (the literal part of itself) *of a living* (a small amount of money). |\n| **“Living” as a noun** | “Living” can mean the money you earn to support yourself. | The joke twists the usual phrase “cost of living

In [17]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to get a *crust* of a living! 🍕😄', 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Personification** | The pizza is treated like a human who can *apply for a job*. | It’s absurd to imagine a slice of dough with cheese and toppings doing a job‑hunt. That absurdity sets the stage for the punchline. |\n| **Wordplay on “crust”** | “Crust” is the outer edge of a pizza. | The joke hinges on the double meaning of the word. |\n| **Play on “cost” / “crust”** | The phrase “a crust of a living” sounds like “a cost of a living” (i.e., the money you need to live). | The pun is that the pizza wants a *crust* (the literal part of itself) *of a living* (a small amount of money). |\n| **“Living” as a noun** | “Living” can mean the money you earn to support yourself. | The joke twists the usual phr

In [18]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to get a *crust* of a living! 🍕😄', 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Personification** | The pizza is treated like a human who can *apply for a job*. | It’s absurd to imagine a slice of dough with cheese and toppings doing a job‑hunt. That absurdity sets the stage for the punchline. |\n| **Wordplay on “crust”** | “Crust” is the outer edge of a pizza. | The joke hinges on the double meaning of the word. |\n| **Play on “cost” / “crust”** | The phrase “a crust of a living” sounds like “a cost of a living” (i.e., the money you need to live). | The pun is that the pizza wants a *crust* (the literal part of itself) *of a living* (a small amount of money). |\n| **“Living” as a noun** | “Living” can mean the money you earn to support yourself. | The joke twists the usual ph

In [19]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti start a band?\n\nBecause it wanted to be a *fettuccine* rockstar—every time it hit a note, it *pasta* the beat!',
 'explanation': '**Why the joke works – a quick word‑play breakdown**\n\n| Part of the joke | What it’s doing | Why it’s funny |\n|------------------|-----------------|----------------|\n| **“Why did the spaghetti start a band?”** | Sets up a classic “why‑does‑X‑do‑Y” question that invites a punchline. | The image of a noodle forming a band is already absurd, priming the reader for a silly answer. |\n| **“Because it wanted to be a *fettuccine* rockstar”** | *Fettuccine* is a type of pasta, but the phrase sounds like “fettuccine rockstar” → “fettuccine” + “rockstar.” | It’s a double‑layered pun: 1) a pasta name, 2) a play on “rockstar.” The absurdity of a noodle aspiring to be a rockstar is the core joke. |\n| **“—every time it hit a note, it *pasta* the beat!”** | “Pasta” is used as a verb, sounding like “pass.” | Two puns 

In [20]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to get a *crust* of a living! 🍕😄', 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Personification** | The pizza is treated like a human who can *apply for a job*. | It’s absurd to imagine a slice of dough with cheese and toppings doing a job‑hunt. That absurdity sets the stage for the punchline. |\n| **Wordplay on “crust”** | “Crust” is the outer edge of a pizza. | The joke hinges on the double meaning of the word. |\n| **Play on “cost” / “crust”** | The phrase “a crust of a living” sounds like “a cost of a living” (i.e., the money you need to live). | The pun is that the pizza wants a *crust* (the literal part of itself) *of a living* (a small amount of money). |\n| **“Living” as a noun** | “Living” can mean the money you earn to support yourself. | The joke twists the usual phr

In [21]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to get a *crust* of a living! 🍕😄', 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Personification** | The pizza is treated like a human who can *apply for a job*. | It’s absurd to imagine a slice of dough with cheese and toppings doing a job‑hunt. That absurdity sets the stage for the punchline. |\n| **Wordplay on “crust”** | “Crust” is the outer edge of a pizza. | The joke hinges on the double meaning of the word. |\n| **Play on “cost” / “crust”** | The phrase “a crust of a living” sounds like “a cost of a living” (i.e., the money you need to live). | The pun is that the pizza wants a *crust* (the literal part of itself) *of a living* (a small amount of money). |\n| **“Living” as a noun** | “Living” can mean the money you earn to support yourself. | The joke twists the usual ph

In [22]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [24]:
workflow.invoke(
    {"messages": [HumanMessage(content="Hello")]},
    config={"configurable": {"thread_id": "1"}}
)

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to get a *crust* of a living! 🍕😄',
 'explanation': '**Explanation of the joke**\n\n> **Why did the pizza apply for a job?  \n>  Because it wanted to get a *crust* of a living! 🍕😄**\n\nThe humor comes from a **word‑play (pun)** that mixes two very different ideas:\n\n| Element | What it normally means | How it’s used in the joke |\n|---------|------------------------|---------------------------|\n| **Pizza “crust”** | The outer, doughy edge of a pizza. | Replaced the word *cost* in the phrase “cost of living.” |\n| **“Cost of living”** | The amount of money needed to cover basic expenses. | The pizza’s “crust” is treated as if it were money, so it “wants a crust of a living.” |\n\n### Step‑by‑step breakdown\n\n1. **Setup** – “Why did the pizza apply for a job?”  \n   - The listener expects a typical “why” answer, but the subject is a pizza, which is obviously not a person. This sets up the expectation t

In [25]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to get a *crust* of a living! 🍕😄', 'explanation': '**Explanation of the joke**\n\n> **Why did the pizza apply for a job?  \n>  Because it wanted to get a *crust* of a living! 🍕😄**\n\nThe humor comes from a **word‑play (pun)** that mixes two very different ideas:\n\n| Element | What it normally means | How it’s used in the joke |\n|---------|------------------------|---------------------------|\n| **Pizza “crust”** | The outer, doughy edge of a pizza. | Replaced the word *cost* in the phrase “cost of living.” |\n| **“Cost of living”** | The amount of money needed to cover basic expenses. | The pizza’s “crust” is treated as if it were money, so it “wants a crust of a living.” |\n\n### Step‑by‑step breakdown\n\n1. **Setup** – “Why did the pizza apply for a job?”  \n   - The listener expects a typical “why” answer, but the subject is a pizza, which is obviously not a person. This sets 

In [26]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1a906d-1427-64bf-8000-18efd1df8c74'}}

In [27]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a906d-1427-64bf-8000-18efd1df8c74'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-09-05T08:50:17.088725+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='eacfae33-b0e9-2c3c-6372-5efab9f7f56c', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to get a *crust* of a living! 🍕😄', 'explanation': '**Explanation of the joke**\n\n> **Why did the pizza apply for a job?  \n>  Because it wanted to get a *crust* of a living! 🍕😄**\n\nThe humor comes from a **word‑play (pun)** that mixes two very different ideas:\n\n| Elemen

In [29]:
response = workflow.invoke(
    {"messages": [HumanMessage(content="Hello, my name is Rahul")]},
    {"configurable": {"thread_id": "1"}}
)

print(response)

{'topic': 'samosa', 'joke': 'Why did the samosa go to therapy?\n\nBecause it felt it was always being *stuffed* with expectations and never got a chance to *crisp* up its own life!', 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Setup: “Why did the samosa go to therapy?”** | A classic “why did X go to therapy?” joke format. | The format primes us for a punchline that will twist a mundane or absurd subject (a samosa) into a human‑like problem. The absurdity of a snack needing therapy already sets a playful tone. |\n| **Wordplay on “stuffed”** | 1. *Stuffed* literally describes a samosa’s filling. 2. “Stuffed with expectations” is a common idiom meaning “overwhelmed by pressure.” | The double meaning lets us read the samosa as both a food item and a metaphor for someone who feels over‑burdened. The pun is the core of the joke’s humor. |\n| **Wordplay on “crisp”** | 1. A samosa is usu

In [30]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to therapy?\n\nBecause it felt it was always being *stuffed* with expectations and never got a chance to *crisp* up its own life!', 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Setup: “Why did the samosa go to therapy?”** | A classic “why did X go to therapy?” joke format. | The format primes us for a punchline that will twist a mundane or absurd subject (a samosa) into a human‑like problem. The absurdity of a snack needing therapy already sets a playful tone. |\n| **Wordplay on “stuffed”** | 1. *Stuffed* literally describes a samosa’s filling. 2. “Stuffed with expectations” is a common idiom meaning “overwhelmed by pressure.” | The double meaning lets us read the samosa as both a food item and a metaphor for someone who feels over‑burdened. The pun is the core of the joke’s humor. |\n| **Wordplay on “crisp”*

In [2]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [3]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [4]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("Step 2 running briefly...")
    time.sleep(2)
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("Step 3 executed")
    return {"done": True}

In [5]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [6]:
try:
    print("Running graph...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": "thread-1"}})
except KeyboardInterrupt:
    print("Graph execution interrupted. The checkpoint is preserved for the resume cell.")

Running graph...
Step 1 executed
Step 2 running briefly...
Step 3 executed


In [7]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}


In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))